# NB3A - Build Golden Data (Kaggle)

Build TRM golden-seed dataset on Kaggle.

Flow: materialize source → install deps → setup env → run `prepare_medreason_seed.py` (train + val) → split goldish/silver/reject with `filter_seed_quality.py` → optionally materialize TRM arrays with `build_trm_dataset.py` → package output as zip.

Notebook is wired against Kaggle IOStream limits:
- Subprocess stdout/stderr is redirected to `<run>/logs/*.log` (no `IOStream.flush timed out` warnings).
- Verbosity of transformers/httpx/OpenAI/tqdm is silenced via env vars.
- Periodic progress pings (log size + elapsed) print every 30 s while a script runs.
- Set `SMOKE_TEST=True` in the setup cell to run on 20 train / 5 val records before scaling up.
- Existing outputs are reused (skip-if-exists) so a re-run after a session timeout does not redo completed stages.

Required Kaggle datasets (attach all under the same owner namespace):
- `minimed-prime-source` (project source)
- `medreason` (raw MedReason JSONL)
- `primekg` (edges.csv + nodes.csv)
- `sapbert`
- `medcpt-query`, `medcpt-article` (optional `medcpt-cross`)

Required Kaggle Secret: `OPENAI_API_KEY` (for `gpt-4o-mini` edge selector).

In [ ]:
from pathlib import Path
import os
import shutil
import sys
import zipfile
from datetime import datetime

SOURCE_ROOT = Path("/kaggle/input/datasets/huynhnhuthuyk18hcm/minimed-prime-source")
WORK_ROOT = Path("/kaggle/working/MiniMed_Prime")

RUN_NAME = ""
OUTPUT_ROOT = Path("/kaggle/working/golden_data")

MEDREASON_SOURCE = "data/medreason"
TRAIN_SPLIT = "train"
VAL_SPLIT = "validation"

# Set True for a fast smoke test (20 train + 5 val) before a full run.
SMOKE_TEST = False

if SMOKE_TEST:
    TRAIN_LIMIT = 20
    VAL_LIMIT = 5
else:
    TRAIN_LIMIT = 500
    VAL_LIMIT = 100
TRAIN_MAX_SAVED = TRAIN_LIMIT
VAL_MAX_SAVED = VAL_LIMIT

EDGE_MAPPER = "auto"
LLM_MODEL_NAME = "gpt-4o-mini"
LLM_DEVICE = "cpu"
RELATION_FILTER = ""
ALLOW_EMPTY_GOLD = False

BUILD_TRM_ARRAYS = True
PACKAGE_OUTPUT_ZIP = True


def resolve_source_dir(root: Path, name: str):
    direct = root / name
    if not direct.is_dir():
        return None

    nested = direct / name
    signatures = {
        "src": "orchestrator.py",
        "scripts": "setup_environment.py",
    }
    sig = signatures.get(name)

    if sig:
        if (direct / sig).exists():
            return direct
        if (nested / sig).exists():
            return nested

    direct_entries = sorted(p.name for p in direct.iterdir())
    if nested.is_dir() and direct_entries == [name]:
        return nested

    return direct


shutil.rmtree(WORK_ROOT, ignore_errors=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

resolved = {}
for name in ["src", "scripts", "notebooks", "tests"]:
    zip_path = SOURCE_ROOT / f"{name}.zip"
    if zip_path.exists():
        with zipfile.ZipFile(zip_path, "r") as archive:
            archive.extractall(WORK_ROOT)
        resolved[name] = f"{zip_path.name} -> extracted"
        continue

    source_dir = resolve_source_dir(SOURCE_ROOT, name)
    if source_dir is not None:
        shutil.copytree(source_dir, WORK_ROOT / name, dirs_exist_ok=True)
        try:
            resolved[name] = str(source_dir.relative_to(SOURCE_ROOT))
        except ValueError:
            resolved[name] = str(source_dir)

for file_name in [
    "requirements-integration.txt",
    "README_KAGGLE_VI.md",
    "TRAINING_KAGGLE_LOCAL_VI.md",
    "IMPLEMENTATION_AUDIT.md",
    "KAGGLE_SOURCE_VERSION.txt",
]:
    source_file = SOURCE_ROOT / file_name
    if source_file.exists():
        shutil.copy2(source_file, WORK_ROOT / file_name)

os.chdir(WORK_ROOT)
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

run_stamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
RUN_NAME = RUN_NAME or f"kaggle_seed_{run_stamp}"
BUILD_ROOT = OUTPUT_ROOT / RUN_NAME
RAW_ROOT = BUILD_ROOT / "raw"
QUALITY_ROOT = BUILD_ROOT / "quality"
TRM_ROOT = BUILD_ROOT / "trm_medical"
LOG_ROOT = BUILD_ROOT / "logs"
for path in [BUILD_ROOT, RAW_ROOT, QUALITY_ROOT, TRM_ROOT, LOG_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

marker = WORK_ROOT / "KAGGLE_SOURCE_VERSION.txt"
print("source root:", SOURCE_ROOT)
print("resolved dirs:", resolved)
print("version marker exists:", marker.exists())
if marker.exists():
    print(marker.read_text(encoding="utf-8"))
print("setup_environment exists:", (WORK_ROOT / "scripts" / "setup_environment.py").exists())
print(f"SMOKE_TEST={SMOKE_TEST} | train_limit={TRAIN_LIMIT} | val_limit={VAL_LIMIT}")
print("run name:", RUN_NAME)
print("build root:", BUILD_ROOT)
print("log root:", LOG_ROOT)

In [ ]:
!pip install rank_bm25

In [ ]:
!nvidia-smi
!python scripts/setup_environment.py --kaggle --skip-downloads

In [ ]:
import os
from pprint import pprint

# Silence noisy logs that overwhelm Kaggle's IOStream and cause
# "IOStream.flush timed out" warnings. Structured JSONL logs under
# /kaggle/working/logs remain intact for diagnosis.
os.environ.setdefault("PYTHONUNBUFFERED", "1")
os.environ.setdefault("PYTHONIOENCODING", "utf-8")
os.environ.setdefault("TQDM_DISABLE", "1")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ.setdefault("OPENAI_LOG", "warning")
os.environ.setdefault("MINIMED_LOG_LEVEL", "WARNING")

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["OPENAI_API_KEY"] = secrets.get_secret("OPENAI_API_KEY")
    print("Loaded OPENAI_API_KEY from Kaggle Secrets.")
except Exception as exc:
    print(f"OPENAI_API_KEY not loaded from Kaggle Secrets: {exc}")

os.environ["MINIMED_JUDGE_MODEL"] = "gpt-4o-mini"
os.environ["MEDREASON_EDGE_LLM"] = LLM_MODEL_NAME
os.environ["MINIMED_SYNTHESIS_MODEL"] = "gpt-4o-mini"

if EDGE_MAPPER in {"auto", "llm"}:
    normalized_model = LLM_MODEL_NAME.strip().lower()
    if normalized_model.startswith("gpt-") and not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("LLM_MODEL_NAME points to OpenAI but OPENAI_API_KEY is missing.")

pprint({
    "OPENAI_API_KEY": "set" if os.environ.get("OPENAI_API_KEY") else "missing",
    "MEDREASON_EDGE_LLM": os.environ.get("MEDREASON_EDGE_LLM"),
    "EDGE_MAPPER": EDGE_MAPPER,
    "LLM_MODEL_NAME": LLM_MODEL_NAME,
    "TQDM_DISABLE": os.environ.get("TQDM_DISABLE"),
    "MINIMED_LOG_LEVEL": os.environ.get("MINIMED_LOG_LEVEL"),
    "TRANSFORMERS_VERBOSITY": os.environ.get("TRANSFORMERS_VERBOSITY"),
})

In [ ]:
from pathlib import Path
from src.utils.kaggle_env import KaggleEnv

logical_paths = [
    "data/kg/primekg",
    "data/checkpoints/sapbert",
    "data/checkpoints/medcpt-query",
    "data/checkpoints/medcpt-article",
]
missing = []
for p in logical_paths:
    resolved_path = KaggleEnv.path(p)
    exists = resolved_path.exists()
    print(p, "->", resolved_path, "exists=", exists)
    if not exists:
        missing.append(p)

if missing:
    raise RuntimeError(f"Missing required datasets: {missing}. Attach them and re-run.")

# Raw MedReason must be discovered explicitly because KaggleEnv.path("data/medreason")
# can false-match the `medreason-8b` model dir (slug substring bug).
INPUT_DATASETS_ROOT = Path("/kaggle/input/datasets")
EXCLUDE_NAME_HINTS = ("medreason-8b",)
MODEL_SIGNATURES = ("config.json",)


def _dir_looks_like_model_checkpoint(path: Path) -> bool:
    return any((path / name).exists() for name in MODEL_SIGNATURES)


def _iter_candidate_dirs(root: Path):
    if not root.exists():
        return
    for owner_dir in root.iterdir():
        if not owner_dir.is_dir():
            continue
        for dataset_dir in owner_dir.iterdir():
            if not dataset_dir.is_dir():
                continue
            versions_dir = dataset_dir / "versions"
            if versions_dir.is_dir():
                for version_dir in sorted(versions_dir.iterdir(), reverse=True):
                    if version_dir.is_dir():
                        yield version_dir
            yield dataset_dir


def _find_medreason_source() -> Path:
    canonical_name = "ours_quality_33000.jsonl"
    canonical_hits = []
    jsonl_candidates = []
    for candidate in _iter_candidate_dirs(INPUT_DATASETS_ROOT):
        lower_name = candidate.name.lower()
        if any(hint in lower_name for hint in EXCLUDE_NAME_HINTS):
            continue
        if _dir_looks_like_model_checkpoint(candidate):
            continue
        if (candidate / canonical_name).exists():
            canonical_hits.append(candidate)
            continue
        jsonl_files = sorted(candidate.glob("*.jsonl"))
        if jsonl_files and "medreason" in lower_name:
            jsonl_candidates.append(candidate)
    if canonical_hits:
        return canonical_hits[0]
    if jsonl_candidates:
        return jsonl_candidates[0]
    attached = [p.name for p in INPUT_DATASETS_ROOT.glob("*/*")] if INPUT_DATASETS_ROOT.exists() else []
    raise RuntimeError(
        "Raw MedReason dataset not found under /kaggle/input/datasets. "
        "Attach the raw MedReason JSONL dataset (e.g. containing ours_quality_33000.jsonl) and re-run. "
        f"Attached dataset dirs seen: {attached}"
    )


MEDREASON_SOURCE_RESOLVED = str(_find_medreason_source())
print("medreason (raw) ->", MEDREASON_SOURCE_RESOLVED)
print("  files:", sorted(p.name for p in Path(MEDREASON_SOURCE_RESOLVED).iterdir())[:10])

In [ ]:
# Install scispacy + en_core_sci_lg on Kaggle Python 3.12.
#
# Two known breakages on current Kaggle images:
#   1. numpy 2.x is pre-installed but thinc/spaCy 3.7 ABI wants numpy<2
#      -> pin numpy<2, scipy<1.13, force-reinstall spacy+thinc.
#   2. scispacy depends on `nmslib`, which has NO Python 3.12 wheel on PyPI
#      and fails to build from source ("metadata-generation-failed").
#      -> try upstream nmslib (binary-only); on failure, fall back to
#         `nmslib-metabrainz`, a maintained fork that ships py312 wheels
#         under the same `nmslib` import name.
#
# IMPORTANT: after this cell finishes, RESTART THE KERNEL
# (Run > Restart & Run All). numpy is already loaded in-process and
# cannot be hot-swapped.

import subprocess
import sys


def pip_install(*args, check: bool = True) -> bool:
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    proc = subprocess.run(cmd)
    ok = proc.returncode == 0
    if not ok and check:
        print(f"[pip] FAILED: {' '.join(args)}")
    return ok


print(">> 1/5 Pinning numpy<2, scipy<1.13 for thinc ABI compat")
pip_install("numpy<2", "scipy<1.13", "--force-reinstall", "--no-deps")

print(">> 2/5 Pinning spaCy 3.7 + thinc 8.2 + blis")
pip_install(
    "spacy>=3.7.4,<3.8", "thinc>=8.2.2,<8.3", "blis<0.8",
    "--force-reinstall", "--no-deps",
)

print(">> 3/5 Installing scispacy pure-Python deps")
pip_install("conllu", "joblib", "pysbd")

print(">> 4/5 Installing nmslib (hnsw index backend for UMLS linker)")
ok_nmslib = pip_install("--only-binary=:all:", "nmslib", check=False)
if not ok_nmslib:
    print("   upstream nmslib has no py312 wheel; trying nmslib-metabrainz fork")
    ok_nmslib = pip_install("nmslib-metabrainz")
if not ok_nmslib:
    print("   [WARN] nmslib install failed → UMLS linker unavailable; "
          "all rows will demote to silver tier.")

print(">> 5/5 Installing scispacy + en_core_sci_lg model")
pip_install("--no-deps", "scispacy==0.5.4")
pip_install(
    "https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/"
    "v0.5.4/en_core_sci_lg-0.5.4.tar.gz"
)

print()
print("=" * 72)
print("scispacy + model install attempted. nmslib ok:", ok_nmslib)
print("→ RESTART KERNEL NOW (Run > Restart & Run All), then re-run notebook.")
print("=" * 72)

In [ ]:
import numpy as np
print("numpy:", np.__version__)
if np.__version__.startswith("2."):
    print(
        "[warn] numpy 2.x detected — thinc/spaCy 3.7 ABI requires numpy<2.\n"
        "       Re-run the install cell above, then RESTART THE KERNEL."
    )

try:
    import spacy
    import scispacy  # noqa: F401
    print("spaCy:", spacy.__version__)
    print("scispaCy:", scispacy.__version__)
except Exception as exc:
    print("failed to import spaCy/scispaCy:", type(exc).__name__, exc)
    print("→ Re-run the install cell then RESTART THE KERNEL.")
    raise

try:
    import nmslib  # noqa: F401
    print("nmslib: ok (UMLS linker will work)")
except Exception as exc:
    print("nmslib import failed:", type(exc).__name__, exc)
    print("  UMLS linker will fall back to rule_based → rows demoted to silver.")

try:
    nlp = spacy.load("en_core_sci_lg")
    print("loaded en_core_sci_lg, pipes:", nlp.pipe_names[:5])
except Exception as exc:
    print("failed to load en_core_sci_lg:", type(exc).__name__, exc)
    print("  goldish tier requires scispacy_umls; rows will be demoted to silver.")

In [ ]:
"""Subprocess runner that keeps Kaggle IOStream happy.

Each pipeline script runs with stdout/stderr streamed to LOG_ROOT/<name>.log.
The notebook cell only receives periodic progress pings and a short tail of
the log on completion. This avoids the "IOStream.flush timed out" warnings
caused by the Kaggle websocket not keeping up with subprocess stdout.
"""
import subprocess
import time
from pathlib import Path


def run_script(name: str, args: list, *, tail_lines: int = 60, poll_sec: int = 30) -> Path:
    log_path = LOG_ROOT / f"{name}.log"
    cmd = ["python", "-u", *[str(a) for a in args]]
    print(f"[run_script] {name}")
    print(f"  cmd : {' '.join(cmd)}")
    print(f"  log : {log_path}")
    started = time.time()
    last_size = 0
    with open(log_path, "w", encoding="utf-8") as fp:
        proc = subprocess.Popen(cmd, stdout=fp, stderr=subprocess.STDOUT, bufsize=1)
        try:
            while True:
                rc = proc.poll()
                if rc is not None:
                    break
                time.sleep(poll_sec)
                try:
                    size = log_path.stat().st_size
                except FileNotFoundError:
                    size = 0
                elapsed = time.time() - started
                print(f"  [{elapsed:7.0f}s] log={size:>9} bytes (+{size - last_size})", flush=True)
                last_size = size
        except KeyboardInterrupt:
            proc.terminate()
            raise
    elapsed = time.time() - started
    print(f"  done in {elapsed:.1f}s, exit={proc.returncode}")
    try:
        lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
        if lines:
            print(f"--- tail({min(tail_lines, len(lines))}) {log_path.name} ---")
            for line in lines[-tail_lines:]:
                print(line)
    except FileNotFoundError:
        pass
    if proc.returncode != 0:
        raise RuntimeError(
            f"{name} failed with exit code {proc.returncode}; full log at {log_path}"
        )
    return log_path


def count_jsonl(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as fp:
        return sum(1 for line in fp if line.strip())


print("run_script helper ready. logs ->", LOG_ROOT)

In [ ]:
train_seed_path = RAW_ROOT / "trm_seed_train.jsonl"
existing = count_jsonl(train_seed_path)
if existing >= TRAIN_MAX_SAVED and TRAIN_MAX_SAVED > 0:
    print(f"[skip] {train_seed_path} already has {existing} records (>= {TRAIN_MAX_SAVED}).")
else:
    if existing:
        print(f"[warn] {train_seed_path} has {existing} records (<{TRAIN_MAX_SAVED}); overwriting from scratch.")
    run_script(
        "prepare_medreason_seed_train",
        [
            "scripts/prepare_medreason_seed.py",
            "--kaggle",
            "--source", MEDREASON_SOURCE_RESOLVED,
            "--split", TRAIN_SPLIT,
            "--output-jsonl", train_seed_path,
            "--edge-mapper", EDGE_MAPPER,
            "--llm-model-name", LLM_MODEL_NAME,
            "--llm-device", LLM_DEVICE,
            "--limit", TRAIN_LIMIT,
            "--max-saved", TRAIN_MAX_SAVED,
        ],
    )
print("train records:", count_jsonl(train_seed_path))

In [ ]:
validation_seed_path = RAW_ROOT / "trm_seed_validation.jsonl"
existing = count_jsonl(validation_seed_path)
if existing >= VAL_MAX_SAVED and VAL_MAX_SAVED > 0:
    print(f"[skip] {validation_seed_path} already has {existing} records (>= {VAL_MAX_SAVED}).")
else:
    if existing:
        print(f"[warn] {validation_seed_path} has {existing} records (<{VAL_MAX_SAVED}); overwriting from scratch.")
    run_script(
        "prepare_medreason_seed_val",
        [
            "scripts/prepare_medreason_seed.py",
            "--kaggle",
            "--source", MEDREASON_SOURCE_RESOLVED,
            "--split", VAL_SPLIT,
            "--output-jsonl", validation_seed_path,
            "--edge-mapper", EDGE_MAPPER,
            "--llm-model-name", LLM_MODEL_NAME,
            "--llm-device", LLM_DEVICE,
            "--limit", VAL_LIMIT,
            "--max-saved", VAL_MAX_SAVED,
        ],
    )
print("validation records:", count_jsonl(validation_seed_path))

In [ ]:
train_quality_dir = QUALITY_ROOT / "train"
train_quality_dir.mkdir(parents=True, exist_ok=True)
run_script(
    "filter_seed_quality_train",
    [
        "scripts/filter_seed_quality.py",
        "--input-jsonl", train_seed_path,
        "--output-dir", train_quality_dir,
    ],
)

In [ ]:
validation_quality_dir = QUALITY_ROOT / "validation"
validation_quality_dir.mkdir(parents=True, exist_ok=True)
run_script(
    "filter_seed_quality_val",
    [
        "scripts/filter_seed_quality.py",
        "--input-jsonl", validation_seed_path,
        "--output-dir", validation_quality_dir,
    ],
)

In [ ]:
validation_quality_dir = QUALITY_ROOT / "validation"
validation_quality_dir.mkdir(parents=True, exist_ok=True)
!python scripts/filter_seed_quality.py \
    --input-jsonl {validation_seed_path} \
    --output-dir {validation_quality_dir}

In [ ]:
if BUILD_TRM_ARRAYS:
    train_goldish = train_quality_dir / "trm_seed_goldish.jsonl"
    val_goldish = validation_quality_dir / "trm_seed_goldish.jsonl"

    train_gold_count = count_jsonl(train_goldish)
    if train_gold_count == 0:
        raise FileNotFoundError(
            f"Train goldish tier is empty at {train_goldish}. "
            "Inspect silver/rejects first before building TRM arrays."
        )
    print(f"Building TRM arrays from train goldish ({train_gold_count} records)")
    run_script(
        "build_trm_dataset_train",
        [
            "scripts/build_trm_dataset.py",
            "--kaggle",
            "--input-jsonl", train_goldish,
            "--output-dir", TRM_ROOT,
            "--split", "train",
        ],
    )

    val_gold_count = count_jsonl(val_goldish)
    if val_gold_count > 0:
        print(f"Building TRM arrays from val goldish ({val_gold_count} records)")
        run_script(
            "build_trm_dataset_val",
            [
                "scripts/build_trm_dataset.py",
                "--kaggle",
                "--input-jsonl", val_goldish,
                "--output-dir", TRM_ROOT,
                "--split", "val",
            ],
        )
    else:
        print("Validation goldish tier empty, skipping val arrays.")

    for split in ["train", "val"]:
        split_dir = TRM_ROOT / split
        print(f"\n=== {split} ===")
        if split_dir.exists():
            print(sorted(p.name for p in split_dir.iterdir()))
        else:
            print("missing")
else:
    print("BUILD_TRM_ARRAYS=False, skipping TRM array materialization.")

In [ ]:
from pathlib import Path

# Structured JSONL logs (per-event, from StructuredLogger)
STRUCTURED_LOGS = Path("/kaggle/working/logs")
print(f"==== STRUCTURED LOGS ({STRUCTURED_LOGS}) ====")
for name in [
    "medreason_adapter.jsonl",
    "layer1_retrieval.jsonl",
    "seed_quality_filter.jsonl",
    "trm_dataset_builder.jsonl",
]:
    path = STRUCTURED_LOGS / name
    print(f"\n=== {name} ===")
    if path.exists():
        print("\n".join(path.read_text(encoding="utf-8").splitlines()[-8:]))
    else:
        print("missing")

# Subprocess stdout/stderr dumps per pipeline stage
print(f"\n\n==== SUBPROCESS LOGS ({LOG_ROOT}) ====")
if LOG_ROOT.exists():
    for path in sorted(LOG_ROOT.glob("*.log")):
        size = path.stat().st_size
        print(f"\n=== {path.name} ({size} bytes, last 8 lines) ===")
        print("\n".join(path.read_text(encoding="utf-8", errors="replace").splitlines()[-8:]))
else:
    print("missing")

In [ ]:
from pathlib import Path

for name in [
    "medreason_adapter.jsonl",
    "layer1_retrieval.jsonl",
    "seed_quality_filter.jsonl",
    "trm_dataset_builder.jsonl",
]:
    path = Path("/kaggle/working/logs") / name
    print(f"\n=== {name} ===")
    if path.exists():
        print("\n".join(path.read_text(encoding="utf-8").splitlines()[-8:]))
    else:
        print("missing")

In [ ]:
import shutil

if PACKAGE_OUTPUT_ZIP:
    archive_base = BUILD_ROOT.parent / BUILD_ROOT.name
    archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=str(BUILD_ROOT))
    print("Packaged archive:", archive_path)
else:
    print("PACKAGE_OUTPUT_ZIP=False, skipping zip.")

print("Final output tree:", BUILD_ROOT)
for child in sorted(BUILD_ROOT.iterdir()):
    print(" -", child)

## Next step

If `goldish` counts look healthy, upload `<RUN_NAME>.zip` or the `quality/` folder as a private Kaggle dataset, attach it to NB3 (`NB3-TRM-Finetune.ipynb`), and point `scripts/train_medical_trm.py` at the extracted `trm_medical/` directory.

If `goldish` is too small, inspect `silver` and `rejects`. Common causes:
- `entity_linker_backend=rule_based` -> scispaCy did not load; re-run the scispaCy install cell.
- `edge_mapper_backend=heuristic` -> LLM backend inactive; check `OPENAI_API_KEY` and `EDGE_MAPPER`.
- `primekg_backend=empty_graph` -> PrimeKG dataset did not resolve; re-run the path-check cell.